# Chapter 2 — The Measurement Instrument

## Question

**We now know what a context bundle is. How do we know what the model actually received?**

Falsifiable version: does the user-visible transcript account for the rendered invocation? If the two totals differ, transcript inspection alone cannot describe the context, and we need an accountable record of the rendered bundle before changing anything.

No model is called. No provider is contacted. The instrument below observes only a synthetic client-side rendered invocation supplied to it. It claims nothing about provider-side additions.

## Setup — one synthetic coding-agent turn

A debugging turn assembled from many sources. Token counts are fixtures, not measurements. Texts are short placeholders; the declared `tokens` field carries the illustrative size. One span is deliberately opaque: the instrument must report it as unknown, never invent it.

In [ ]:
import hashlib
from dataclasses import dataclass, field

@dataclass
class RecordItem:
    id: str
    label: str
    source: str       # which subsystem placed this item
    type: str         # kind of material
    position: int = -1
    age: int = 0      # turns since the item entered the bundle
    tokens: int = 0   # fixture count; None means unobserved
    authority: str = ''
    scope: str = ''
    repetition: str = ''   # 'unique' | 'repeats:<id>' | 'unknown'
    stable: str = ''       # decided across turns, filled in later
    cache_span: str = ''
    verbatim: str = ''     # sha256 of exact bytes, or '' when opaque
    opaque: bool = False

def sha(b: str) -> str:
    return hashlib.sha256(b.encode('utf-8')).hexdigest()[:16]

turn_n = [
    RecordItem('sys', 'System instructions', 'harness', 'instruction', tokens=800, authority='vendor', scope='global', age=5, cache_span='prefix-A'),
    RecordItem('proj', 'Project instructions', 'harness', 'instruction', tokens=1900, authority='project', scope='project', age=5, cache_span='prefix-A'),
    RecordItem('tool_lint', 'Tool definition: lint', 'harness', 'tool_definition', tokens=900, authority='harness', scope='task', age=5, cache_span='prefix-A'),
    RecordItem('tool_search', 'Tool definition: search', 'harness', 'tool_definition', tokens=1400, authority='harness', scope='task', age=5, cache_span='prefix-A'),
    RecordItem('tool_run', 'Tool definition: run_tests', 'harness', 'tool_definition', tokens=2900, authority='harness', scope='task', age=5, cache_span='prefix-A'),
    RecordItem('history', 'Conversation history turns 1-4', 'harness', 'history', tokens=9800, authority='mixed', scope='task', age=1, cache_span='tail'),
    RecordItem('summary', 'Summary written at turn 4', 'summariser', 'summary', tokens=900, authority='derived', scope='task', age=1, cache_span='tail'),
    RecordItem('file_a', 'checkout.py first read', 'retriever', 'file', tokens=3900, authority='evidence', scope='project', age=2, cache_span='tail'),
    RecordItem('file_b', 'checkout.py second read (80% overlap)', 'retriever', 'file', tokens=3200, authority='evidence', scope='project', age=0, cache_span='tail'),
    RecordItem('res_tests', 'Tool result: test output', 'tool', 'tool_result', tokens=3100, authority='data', scope='observation', age=0, cache_span='tail'),
    RecordItem('res_search', 'Tool result: search hits', 'tool', 'tool_result', tokens=1800, authority='data', scope='observation', age=1, cache_span='tail'),
    RecordItem('user_msg', 'Current user message', 'user', 'request', tokens=120, authority='user', scope='turn', age=0, cache_span='tail'),
    RecordItem('env', 'Environment state (cwd, os, date)', 'runtime', 'environment', tokens=480, authority='fact', scope='session', age=0, cache_span='tail'),
    RecordItem('provider_side', 'Provider-side span (not visible to harness)', 'provider', 'unknown', tokens=None, authority='unknown', scope='unknown', opaque=True),
]

# Exact bytes for the observable items (short placeholders stand in for large contents).
bytes_n = {
    'sys': 'SYSTEM: you are a coding agent.',
    'proj': 'PROJECT: follow the repo testing rules.',
    'tool_lint': 'TOOL lint(file): run the linter.',
    'tool_search': 'TOOL search(query): search the repo.',
    'tool_run': 'TOOL run_tests(target): run the test suite.',
    'history': 'HISTORY: turns 1-4 with four failed tool calls.',
    'summary': 'SUMMARY: tried X, failed with assertion at line 41.',
    'file_a': 'FILE checkout.py read 1.',
    'file_b': 'FILE checkout.py read 2, overlapping read 1.',
    'res_tests': 'RESULT: 3 failures, assertion mismatch.',
    'res_search': 'RESULT: 6 hits for checkout total.',
    'user_msg': 'Why does the checkout test still fail?',
    'env': 'ENV: /repo, linux, 2026-09-24.',
}
print(f'{len(turn_n)} records defined ({sum(1 for r in turn_n if r.opaque)} opaque).')

In [ ]:
# Render: assign positions in order, hash exact bytes. Opaque items keep no hash.
def render(records, byte_map):
    for pos, r in enumerate(records):
        r.position = pos
        if not r.opaque:
            r.verbatim = sha(byte_map[r.id])
    return records

render(turn_n, bytes_n)
for r in turn_n:
    tok = 'unknown' if r.tokens is None else str(r.tokens)
    print(f"{r.position:2d}  {r.id:11s} {tok:>7s}  {r.source:9s} {r.type:15s} {r.verbatim or '(unobserved)'}")

## Baseline — transcript versus rendered invocation

The visible transcript is the user and assistant turns. Everything else was placed by software.

In [ ]:
transcript_user_prior = 380    # earlier user turns, visible in the chat log
transcript_assistant = 640   # earlier assistant replies, visible in the chat log
transcript_current = next(r for r in turn_n if r.id == 'user_msg').tokens
visible_transcript = transcript_user_prior + transcript_assistant + transcript_current
rendered_total = sum(r.tokens for r in turn_n if not r.opaque)
print(f'visible transcript tokens   {visible_transcript}')
print(f'rendered context tokens     {rendered_total}')
print(f'difference (hidden stack)   {rendered_total - visible_transcript}')
print(f'user message share          {transcript_current / rendered_total:.2%}')
assert rendered_total == 31200
assert visible_transcript < 0.05 * rendered_total, 'transcript should be a small minority'

## Per-source accounting — every token attributed, totals reconciled

In [ ]:
groups = {
    'instructions': ['sys', 'proj'],
    'tools (definitions)': ['tool_lint', 'tool_search', 'tool_run'],
    'history': ['history', 'summary'],
    'files': ['file_a', 'file_b'],
    'tool results': ['res_tests', 'res_search'],
    'user request': ['user_msg'],
    'environment': ['env'],
}
by_id = {r.id: r for r in turn_n}
category_totals = {name: sum(by_id[i].tokens for i in ids) for name, ids in groups.items()}
for name, total in category_totals.items():
    print(f'{name:22s} {total:6d}  ({total / rendered_total:5.1%})')
print(f"{'TOTAL':22s} {sum(category_totals.values()):6d}")
assert sum(category_totals.values()) == rendered_total, 'categories must reconcile with the invocation total'
tool_def_overhead = category_totals['tools (definitions)']
print(f'\nTool-definition overhead before any call: {tool_def_overhead} tokens ({tool_def_overhead / rendered_total:.1%})')

## Repetition — counted, not removed

The second file read overlaps the first by 80% (fixture fact: 2560 of its 3200 tokens). Counting duplication is not permission to prune; Chapter 10 owns removal.

In [ ]:
overlap_tokens = 2560  # declared fixture property of file_b relative to file_a
by_id['file_b'].repetition = 'repeats:file_a (2560 tokens)'
for r in turn_n:
    if not r.repetition:
        r.repetition = 'unique'
print(f'repeated tokens in bundle: {overlap_tokens} ({overlap_tokens / rendered_total:.1%} of rendered)')
print(f"file_b record: {by_id['file_b'].repetition}")
assert overlap_tokens == int(0.8 * by_id['file_b'].tokens)
assert overlap_tokens < rendered_total

## Stability across two turns — what survived byte-identical

Turn N+1: instructions and tool definitions re-sent unchanged; environment re-rendered (new timestamp, same size); history appended; one new tool result and one new user message arrive. No provider caching is implemented; this is measurement only.

In [ ]:
import copy
bytes_np1 = dict(bytes_n)
bytes_np1['env'] = 'ENV: /repo, linux, 2026-09-24T09:41Z.'  # re-rendered timestamp
bytes_np1['history'] = bytes_n['history'] + ' + turn-5 exchange.'  # appended
bytes_np1['res_lint'] = 'RESULT: lint clean on checkout.py.'       # newly arrived
bytes_np1['user_msg2'] = 'Run the focused test again.'              # newly arrived

turn_np1 = copy.deepcopy([r for r in turn_n if not r.opaque])
for r in turn_np1:
    if r.id == 'history':
        r.tokens = 11300
        r.age = 0
by_id_np1_extra = [
    RecordItem('res_lint', 'Tool result: lint output', 'tool', 'tool_result', tokens=2200, authority='data', scope='observation', age=0, cache_span='tail'),
    RecordItem('user_msg2', 'Follow-up user message', 'user', 'request', tokens=140, authority='user', scope='turn', age=0, cache_span='tail'),
]
# Insert the new items just before the environment/user tail, mimicking harness assembly.
order_np1 = [r for r in turn_np1 if r.id not in ('env', 'user_msg')] + by_id_np1_extra + [r for r in turn_np1 if r.id in ('env', 'user_msg')]
render(order_np1, bytes_np1)
for r in order_np1:
    old = bytes_n.get(r.id)
    r.stable = 'stable' if (old is not None and bytes_np1.get(r.id) == old) else 'dynamic'

stable_prefix = []
for a, b in zip(turn_n, order_np1):
    if a.id == b.id and a.verbatim == b.verbatim:
        stable_prefix.append(a)
    else:
        break
stable_prefix_tokens = sum(r.tokens for r in stable_prefix)
print(f"stable prefix items: {[r.id for r in stable_prefix]}")
print(f'stable prefix tokens: {stable_prefix_tokens}')
print(f"changed items: {[r.id for r in order_np1 if r.stable == 'dynamic' and r.id in bytes_n]}")
print(f"new items: {[r.id for r in order_np1 if r.id not in bytes_n]}")
assert [r.id for r in stable_prefix] == ['sys', 'proj', 'tool_lint', 'tool_search', 'tool_run']
assert stable_prefix_tokens == 7900

## Exact identity — looks the same is not byte-identical

In [ ]:
def normalised(s: str) -> str:
    return ' '.join(s.split())

a = 'Deploy on Friday. '
b = 'Deploy on Friday.'
print('normalised equal:', normalised(a) == normalised(b))
print('byte-identical:  ', sha(a) == sha(b))
assert normalised(a) == normalised(b)
assert sha(a) != sha(b), 'stability claims must rest on bytes, not impressions'

## Instrument failure, handled honestly — the opaque span

In [ ]:
opaque = [r for r in turn_n if r.opaque]
for r in opaque:
    print(f'{r.id}: tokens=unknown, bytes=unobserved, verbatim=(none)')
    r.repetition = 'unknown'
assert all(r.tokens is None and r.verbatim == '' for r in opaque)
assert sum(r.tokens for r in turn_n if not r.opaque) == rendered_total
print('Visible blindness is better than fabricated completeness: totals cover only observed spans.')

## Observation — Context Measurement Report (synthetic)

In [ ]:
print('CONTEXT MEASUREMENT REPORT (synthetic client-side capture)')
print(f'total rendered tokens:      {rendered_total}')
for name, total in category_totals.items():
    print(f'  {name:22s} {total:6d}')
print(f'repeated tokens:            {overlap_tokens}')
print(f'stable prefix tokens (N->N+1): {stable_prefix_tokens}')
print(f'tool-definition overhead:   {tool_def_overhead}')
print(f"tool-result contribution:   {category_totals['tool results']} (turn N) + 2200 new (turn N+1)")
print(f"user-message contribution:  {category_totals['user request']} ({category_totals['user request'] / rendered_total:.2%})")
print('unknown/opaque spans:       1 (provider_side, excluded from totals)')

## Try it

1. Change `overlap_tokens` and re-run the report — watch repetition move while nothing is removed.
2. Alter `bytes_np1['env']` back to the turn-N text and re-run stability — the stable prefix still stops at `history`, because history grew.
3. Add a second opaque record and confirm the reconciliation assertion still holds (opaque spans never enter totals).

In [ ]:
# Reader scratch space (commented out so Run All stays at baseline):
# overlap_tokens = 1000
# print('repeated share:', overlap_tokens / rendered_total)

## What this demonstrates

- Before changing context, we need an accountable representation of the rendered bundle: per-item source, type, position, tokens, authority, scope, repetition, stability, and exact-byte identity.
- The transcript is a minority view: here the visible transcript is under 5% of the 31,200 rendered fixture tokens.
- Repetition and stability are countable without being acted upon; opaque spans are reported as unknown.

## What this does not demonstrate

- That any material is unnecessary, or that repeated material is safe to delete.
- That fewer tokens improve behaviour or lower real cost.
- That observed client context equals invisible provider-side context.
- That cache reuse occurred, or that any optimisation is justified.

## Connection to the chapter

This is Context Lab v0 in miniature: observe only, reconcile totals, freeze baselines. Measuring is not improving. With the bundle now visible, one fact dominates the tables and demands explanation:

> If the user's message is only one part of the measured bundle, what produced everything else?

That is Chapter 3.